In [1]:
## Import Modules
import pandas as pd
import numpy as np
import csv



In [2]:
# Read CSV and convert to NumPy array
review_df = pd.read_csv('rotten_tomatoes_critic_reviews.csv')
review_df = review_df[['rotten_tomatoes_link','review_content']]

review_df = review_df.dropna()
review_df_condensed = review_df.groupby('rotten_tomatoes_link', as_index=False)['review_content'].agg(' '.join)

In [ ]:
print(review_df_condensed.head(5))
print('--------------------')
print(review_df_condensed.loc[0, 'review_content'])


In [4]:
movie_df = pd.read_csv('rotten_tomatoes_movies.csv')
movie_df = movie_df[['rotten_tomatoes_link','movie_title']]
movie_df_unique = movie_df.drop_duplicates(subset="rotten_tomatoes_link")

movie_and_reviews_df = review_df_condensed.merge(movie_df_unique, on="rotten_tomatoes_link", how="inner")
movie_and_reviews_df = movie_and_reviews_df[["movie_title", "review_content"]]


In [ ]:
print(movie_and_reviews_df.head(10))

print('--------------------')
print(movie_and_reviews_df.loc[0, 'review_content'])

In [6]:
oscar_df = pd.read_csv('ML_dataset.csv')
oscar_df = oscar_df[['title','best_picture_nom']]
oscar_df_unique = oscar_df.drop_duplicates(subset="title")


movie_and_reviews_df.rename(columns={'movie_title': 'title'}, inplace=True)

master_df = movie_and_reviews_df.merge(oscar_df_unique, on="title", how="inner")

In [ ]:
print(master_df.head(50))

In [8]:
#format an np array of tuples (review, label)
master_np = list(
    master_df[["best_picture_nom", "review_content"]]
    .itertuples(index=False, name=None)
)
print(master_np[:5])

[(0, 'A fantasy adventure that fuses Greek mythology to contemporary American places and values. Anyone around 15 (give or take a couple of years) will thrill to the visual spectacle Uma Thurman as Medusa, the gorgon with a coiffure of writhing snakes and stone-inducing hypnotic gaze is one of the highlights of this bewitching fantasy With a top-notch cast and dazzling special effects, this will tide the teens over until the next Harry Potter instalment. Whether audiences will get behind The Lightning Thief is hard to predict. Overall, it\'s an entertaining introduction to a promising new world -- but will the consuming shadow of Potter be too big to break free of? What\'s really lacking in The Lightning Thief is a genuine sense of wonder, the same thing that brings viewers back to Hogwarts over and over again. It\'s more a list of ingredients than a movie-magic potion to enjoy from start to finish. Harry Potter knockoffs don\'t come more transparent and slapdash than this wannabe-fran

In [51]:
def load_feature_dictionary(file):
    """
    Creates a map of words to vectors using the file that has the glove
    embeddings.

    Parameters:
        file (str): File path to the glove embedding file.

    Returns:
        A dictionary indexed by words, returning the corresponding glove
        embedding np.ndarray.
    """
    glove_map = dict()
    with open(file, encoding='utf-8') as f:
        read_file = csv.reader(f, delimiter='\t')
        for row in read_file:
            word, embedding = row[0], row[1:]
            glove_map[word] = np.array(embedding, dtype=float)
    return glove_map

In [ ]:
final_text_output = ""
        for review in curr_tsv:
            sum_glove_vectors = np.zeros(300)
            review_text = review[1]
            valid_word_count = 0
            for word in review_text.split():  # for iterating thru each word in the review
                if word in feature_dict.keys(): # if word is in GloVe dict
                    valid_word_count +=1
                    word_vector = feature_dict.get(word) # retrieve vector associated with the word
                    sum_glove_vectors += word_vector
            sum_glove_vectors *= 1/valid_word_count
            
            # concatenate all labels + features into correct format
            label = f"{review[0]:.6f}"
            final_text_output += str(label) + '\t' # first thing in each row is label

            for j in range(len(sum_glove_vectors)-1):
                curr_rounded_sum = f"{sum_glove_vectors[j]:.6f}"
                final_text_output += str(curr_rounded_sum) + '\t'

            final_sum = sum_glove_vectors[len(sum_glove_vectors)-1]
            final_rounded_sum = f"{final_sum:.6f}"
            final_text_output += str(final_rounded_sum) + '\n'

        with open(output_files[i], "w") as file:
            file.write(final_text_output)